# 04 · Evaluation and error analysis

The brief asks for more than one headline number: overall metrics, results by group, failure cases, latency and whether the model is usable.

Both models are tested on **both** test sets, so the comparison is fair:
- Roboflow test: the brief's original benchmark
- Merged test: includes Indian roads, closer to real use in India

In [ ]:
# --- Setup: works locally and on Google Colab (free T4 GPU) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Upload the project folder to Google Drive first (MyDrive/Road-Damage-Pothole-Detection)
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/Road-Damage-Pothole-Detection")
    !pip -q install -r requirements.txt
else:
    # run from the notebooks/ folder -> go to the project root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Project root:", os.getcwd())

In [ ]:
MODELS = 'models/baseline_yolov8n.pt models/advanced_yolov8s_india.pt'
!python -m pothole.evaluate --models {MODELS} --data data/processed/roboflow/data.yaml
!python -m pothole.evaluate --models {MODELS} --data data/processed/merged/data.yaml

In [ ]:
from IPython.display import Markdown, display
for tag in ['roboflow', 'merged']:
    p = f'reports/eval/{tag}/comparison.md'
    if os.path.exists(p): display(Markdown(f'### Test set: {tag}\n' + open(p).read()))

## Failure cases
Green = ground truth (thick green = missed), blue = correct detection, red = false alarm.

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob("reports/eval/merged/advanced_yolov8s_india/errors/*.jpg"))[:8]:
    print(os.path.basename(p)); display(Image(p, width=700))

## Choosing the confidence threshold
For inspection, missing a pothole costs more than a false alarm (a person checks every alarm anyway), so pick a threshold that favours **recall**.

In [ ]:
import json, subprocess
rows = []
for c in [0.1, 0.15, 0.25, 0.35, 0.5]:
    subprocess.run([sys.executable, "-m", "pothole.evaluate", "--models", "models/advanced_yolov8s_india.pt",
                    "--data", "data/processed/merged/data.yaml", "--conf", str(c), "--n-errors", "0",
                    "--tag", f"conf_{c}"], check=True, capture_output=True)
    m = json.load(open(f"reports/eval/conf_{c}/advanced_yolov8s_india/metrics.json"))
    rows.append({"conf": c, "precision": m["precision_at_conf"], "recall": m["recall_at_conf"],
                 "f1": m["f1_at_conf"], "FP": m["FP"], "FN": m["FN"]})
import pandas as pd; pd.DataFrame(rows)

## Is it usable? (fill in)

- **Overall:** mAP@0.5 = … on Indian roads; recall at the chosen threshold = …
- **Weak spots:** small/far potholes (recall by size), water-filled potholes, night/shadow images …
- **Speed:** … ms/image on CPU → about … images/s, enough for a phone dashcam at 1-2 frames/s
- **Verdict:** suitable to *rank roads for inspection*, not to certify a road as safe.
- **Responsible use:** every flagged pothole is checked by a person before repair orders; "no detection" does not mean "no pothole".